# Install Unsloth Library

<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="100"></a></a> Easily finetune & train LLMs and Get faster with unsloth
 <i><a href="https://github.com/unslothai/unsloth">Github</a> </i>
</div>

Unsloth provides beginner friendly notebooks for training LLMs on Colab, Kaggles with less than 80% necessary memories using techniques like Quantization, PEFT, RoPE Scaling for long context, ...

In [1]:
%%capture
# %%capture for suppressing the install log
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git""

##  Setting up LLaMA 3.2 model

In [2]:
# Models Loading
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! Unsloths auto supports RoPE Scaling internally!
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B", # or choose "unsloth/Llama-3.2-1B" # base version of Llama-3.2
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.10.1: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.4.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

In [3]:
# PEFT settings
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "embed_tokens", "lm_head",], # Add for continual pretraining],
    lora_alpha = 32, # Suggested rx2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Offloading input_embeddings to disk to save VRAM


/usr/local/lib/python3.10/dist-packages/unsloth/models/_utils.py:887: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  offloaded_W = torch.load(filename, map_location = "cpu", 

Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2024.10.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Casting embed_tokens to float32
Unsloth: Casting lm_head to float32


# Step 1: Continual Pretraining

Teach the model "learn" the language

### Test Model's Vietnamese Completion

In [4]:
# Inference Testing
## Inference Mode
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    "Hồ Chí Minh"
], return_tensors = "pt").to("cuda")


outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

['Hồ Chí Minh City, Vietnam, 2018-12-04 - (ACN Newswire) - The 2018 Vietnam International Travel Mart (VITM) was held from 29 November to 2 December 2018 at the Saigon Exhibition and Convention Center (SECC) in Ho Chi Minh City, Vietnam']

### Data Preps

We now use the Vietnamese subset of the [Wikipedia dataset](https://huggingface.co/datasets/wikimedia/wikipedia) to first continually pretrain the model.

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

**[NOTE]** Remember to check the Llama 3.2 Pretrained prompts in [here](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_1/#-pretrained-model-prompt-)

In [5]:
from datasets import load_dataset

# We will download the dataset at 2023-11-01
dataset = load_dataset("wikimedia/wikipedia", "20231101.vi", split = "train",)
# Ignore red progress bar

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

train-00000-of-00004.parquet:   0%|          | 0.00/291M [00:00<?, ?B/s]

train-00001-of-00004.parquet:   0%|          | 0.00/71.0M [00:00<?, ?B/s]

train-00002-of-00004.parquet:   0%|          | 0.00/50.9M [00:00<?, ?B/s]

train-00003-of-00004.parquet:   0%|          | 0.00/316M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1288680 [00:00<?, ? examples/s]

In [6]:
dataset

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 1288680
})

In [7]:
# We select 0.5% of the data to make training faster!
dataset = dataset.train_test_split(train_size = 0.005, shuffle=True, seed=4)["train"]

In [8]:
# Format
EOS_TOKEN = tokenizer.eos_token # end with eos
def formatting_prompts_func(examples):
    return { "text" : [example + EOS_TOKEN for example in examples["text"]] }

dataset = dataset.map(formatting_prompts_func, batched = True,)


Map:   0%|          | 0/6443 [00:00<?, ? examples/s]

In [9]:
# Print checking
for row in dataset[:3]["text"]:
    print("=========================")
    print(row)

Hoàng Liên Sơn là một tỉnh cũ thuộc vùng Tây Bắc Bộ Việt Nam.

Địa lý
Tỉnh Hoàng Liên Sơn nằm ở giữa đông và tây Bắc Bộ, có vị trí địa lý:
Phía bắc giáp tỉnh Vân Nam (Trung Quốc)
Phía nam giáp tỉnh Sơn La
Phía đông giáp tỉnh Hà Tuyên và tỉnh Vĩnh Phú
Phía tây giáp tỉnh Lai Châu.

Lịch sử
Tỉnh Hoàng Liên Sơn được thành lập vào ngày 27 tháng 12 năm 1975, trên cơ sở sáp nhập các tỉnh Lào Cai, Yên Bái và các huyện Mù Căng Chải, Văn Chấn, Trạm Tấu, Than Uyên của tỉnh Nghĩa Lộ (riêng 2 huyện Bắc Yên và Phù Yên trở về tỉnh Sơn La quản lý).

Khi hợp nhất, tỉnh Hoàng Liên Sơn có 4 thị xã: Lào Cai, Yên Bái, Cam Đường, Nghĩa Lộ và 16 huyện: Bắc Hà, Bảo Thắng, Bảo Yên, Bát Xát, Lục Yên, Mù Cang Chải, Mường Khương, Sa Pa, Si Ma Cai, Than Uyên, Trạm Tấu, Trấn Yên, Văn Bàn, Văn Chấn, Văn Yên, Yên Bình. Tỉnh lỵ của tỉnh ban đầu được đặt tại thị xã Lào Cai, đến năm 1978 được dời về thị xã Yên Bái.

Năm 1978, chuyển thị xã Nghĩa Lộ thành thị trấn Nghĩa Lộ thuộc huyện Văn Chấn. Năm 1979, sáp nhập thị xã 

<a name="Train"></a>
### Continual Pretraining
Now let's use Unsloth's `UnslothTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 120 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

Also set `embedding_learning_rate` to be a learning rate at least 2x or 10x smaller than `learning_rate`.

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

# training mode
FastLanguageModel.for_training(model)

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 8,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        max_steps = 120,
        warmup_steps = 10,
        # warmup_ratio = 0.1,
        # num_train_epochs = 1,

        warmup_ratio = 0.1,
        num_train_epochs = 1,

        learning_rate = 5e-5, # you should choose this carefully ..
        embedding_learning_rate = 5e-6,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Map (num_proc=8):   0%|          | 0/6443 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [11]:
trainer_stats = trainer.train()

Unsloth: Setting lr = 5.00e-06 instead of 5.00e-05 for embed_tokens.
Unsloth: Setting lr = 5.00e-06 instead of 5.00e-05 for lm_head.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 6,443 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 8
\        /    Total batch size = 16 | Total steps = 120
 "-____-"     Number of trainable parameters = 812,318,720


Step,Training Loss
1,2.209000
2,2.346700
3,2.295800
4,2.120100
5,2.281100
6,2.071600
7,1.893700
8,2.144900
9,2.095800
10,1.952000


### Model Inference Test After continual pretraining

In [12]:
# Inference Testing
## Inference Mode
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    "Hồ Chí Minh"
], return_tensors = "pt").to("cuda")


outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

## Hallucination but it's now perfer Vietnamese

['Hồ Chí Minh là một thành phố thuộc tỉnh Quảng Nam, Việt Nam.\n\nXem thêm \n\nThành phố Hồ Chí Minh\n\nThành phố Hồ Chí Minh\n\nThành phố Hồ Chí Minh\n\nThành phố Hồ Chí Minh\n\nThành phố Hồ Chí Minh\n\nThành phố Hồ Chí Minh\n\nThành phố Hồ Chí Minh\n\n']

In [13]:
inputs = tokenizer(
[
    "Đại học Bách Khoa"
], return_tensors = "pt").to("cuda")


outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

## Hallucination?

['Đại học Bách Khoa Hà Nội là một trường đại học công lập ở Hà Nội, Việt Nam. Trường được thành lập năm 1901, là trường đại học lâu đời nhất Việt Nam. Trường có 3 cơ sở chính: Đại học Bách Khoa Hà Nội, Đại học Bách Khoa Hà Nội - Đại học Quốc gia Hà']

## Step 2: Instruction Finetuning

Teach model how to "follow" instruction

We will continue Instruction Finetuning with current PEFT weights (in practice, you should choose different configuration for this stage)

### Test Model's Ability  for Instruction Following

In [14]:
# Inference Testing

inputs = tokenizer(
[
    "Nhiệm vụ của thủ tướng là gì?"
], return_tensors = "pt").to("cuda")


outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

## Just completion?

['Nhiệm vụ của thủ tướng là gì? Thủ tướng là người đứng đầu chính phủ nước nào? Thủ tướng là người đứng đầu chính phủ nước nào? Thủ tướng là người đứng đầu chính phủ nước nào? Thủ tướng là người đứng đầu chính phủ nước nào? Thủ tướng là người đứng đầu chính phủ nước nào? Thủ tướng là người đứng đầu chính phủ nước']

### Data Preps

We now use the classical dataset [Vi-Alpaca-BKAI](https://huggingface.co/datasets/bkai-foundation-models/vi-alpaca) to fine-tuning the models


**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

**[NOTE]** Similar to Pretraining Stage please check the instruct prompt of Llama3.2 [here](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_1/#-instruct-model-prompt-). However we will use our alpaca prompt format

In [15]:
from datasets import load_dataset
alpaca_dataset = load_dataset("bkai-foundation-models/vi-alpaca", split = "train")

README.md:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

(…)-00000-of-00001-b0855b79e84114ca.parquet:   0%|          | 0.00/26.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50006 [00:00<?, ? examples/s]

In [16]:
print(alpaca_dataset[0])

{'instruction': 'Hãy viết một bài blog ngắn về lợi ích của việc đọc sách.', 'input': 'Tiêu đề: Lợi ích của việc đọc sách\nMô tả: Bài blog ngắn này sẽ giải thích những lợi ích mà việc đọc sách mang lại cho con người.', 'output': 'Bài viết: \nViệc đọc sách có rất nhiều lợi ích cho con người. Đây là một hoạt động giáo dục và giải trí hữu ích, giúp chúng ta mở rộng kiến thức và hiểu biết về thế giới xung quanh.\n\nMột trong những lợi ích đáng kể của việc đọc sách là cải thiện khả năng ngôn ngữ của chúng ta. Khi đọc sách, chúng ta tiếp xúc với các từ ngữ mới, cấu trúc câu phức tạp và ngữ cảnh sử dụng. Điều này giúp chúng ta mở rộng vốn từ vựng và cải thiện khả năng diễn đạt bằng ngôn ngữ.\n\nNgoài ra, đọc sách cũng có tác động tích cực đến trí tuệ và tư duy của con người. Việc đọc sách đòi hỏi chúng ta tập trung, tư duy logic và sáng tạo. Chúng ta phải tưởng tượng và hình dung các tình huống, nhân vật và cốt truyện. Điều này giúp phát triển não bộ và khả năng tư duy sáng tạo của chúng ta.\n

In [17]:
alpaca_prompt = """Dưới đây là hướng dẫn mô tả một nhiệm vụ. Viết một phản hồi hoàn thành yêu cầu một cách thích hợp.

### Nhiệm vụ:
{}

### Đầu vào:
{}

### Câu trả lời:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

alpaca_dataset = alpaca_dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/50006 [00:00<?, ? examples/s]

<a name="Train"></a>
### Instruction Finetuning

We again employ `UnslothTrainer` and do instruction finetuning!

In [18]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments


# training mode
FastLanguageModel.for_training(model)

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = alpaca_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 8,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        # Use num_train_epochs and warmup_ratio for longer runs!
        max_steps = 120,
        warmup_steps = 10,
        # warmup_ratio = 0.1,
        # num_train_epochs = 1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-5,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.00,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Map (num_proc=8):   0%|          | 0/50006 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [19]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 50,006 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 8
\        /    Total batch size = 16 | Total steps = 120
 "-____-"     Number of trainable parameters = 812,318,720


Unsloth: Setting lr = 1.00e-05 instead of 5.00e-05 for embed_tokens.
Unsloth: Setting lr = 1.00e-05 instead of 5.00e-05 for lm_head.


Step,Training Loss
1,1.793900
2,1.761400
3,1.790500
4,1.688800
5,1.842200
6,1.602800
7,1.583300
8,1.990500
9,1.468400
10,1.584400


### Inference again

In [20]:
# Inference Testing
## Inference Mode
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Nhiệm vụ của thủ tướng là gì?", # instruction
        "", # input - leave blank if there is no input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")


from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
outputs = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Dưới đây là hướng dẫn mô tả một nhiệm vụ. Viết một phản hồi hoàn thành yêu cầu một cách thích hợp.

### Nhiệm vụ:
Nhiệm vụ của thủ tướng là gì?

### Đầu vào:


### Câu trả lời:
Thủ tướng là người đứng đầu chính phủ của một quốc gia hoặc vùng lãnh thổ. Thủ tướng có trách nhiệm lãnh đạo và điều hành chính phủ, đảm bảo việc thực hiện các chính sách và quản lý các hoạt động của quốc gia. Thủ tướng thường được bầu cử hoặc bổ nhiệm theo quy định của pháp luật và có quyền lực và trách nhiệm cao trong việc điều hành và quản lý quốc gia.<|end_of_text|>


In [21]:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

['Dưới đây là hướng dẫn mô tả một nhiệm vụ. Viết một phản hồi hoàn thành yêu cầu một cách thích hợp.\n\n### Nhiệm vụ:\nNhiệm vụ của thủ tướng là gì?\n\n### Đầu vào:\n\n\n### Câu trả lời:\nThủ tướng là người đứng đầu chính phủ của một quốc gia hoặc vùng lãnh thổ. Thủ tướng có trách nhiệm lãnh đạo và điều hành chính phủ, đảm bảo việc thực hiện các chính sách và quản lý các hoạt động của quốc gia. Thủ tướng thường được bầu cử hoặc bổ nhiệm theo quy định của pháp luật và có quyền lực và trách nhiệm cao trong việc điều hành và quản lý quốc gia.']

In [22]:
# Another testing
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Dịch câu sau sang tiếng Việt:", # instruction
        "Hello, my name is Duck", # input - leave blank if there is no input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")


from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
outputs = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Dưới đây là hướng dẫn mô tả một nhiệm vụ. Viết một phản hồi hoàn thành yêu cầu một cách thích hợp.

### Nhiệm vụ:
Dịch câu sau sang tiếng Việt:

### Đầu vào:
Hello, my name is Duck

### Câu trả lời:
Xin chào, tôi tên là Duck<|end_of_text|>


# Next Steps

1. Save and load continual pretrained and fine-tuned models? Refer to [2]

2. More advanced libraries for fine-tuning LLMs? [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory)

3. Try hard? [llama-recipes](https://github.com/meta-llama/llama-recipes)


# References

[1] [Continued pretraining - Korean + Unsloth.ipynb](https://colab.research.google.com/drive/1tEd1FrOXWMnCU9UIvdYhs61tkxdMuKZu?usp=sharing#scrollTo=1oNjUwxOyG8C)

[2] [Llama-3.2 1b + 3b + Unsloth 2x faster finetuning.ipynb](https://colab.research.google.com/drive/1hoHFpf7ROqk_oZHzxQdfPW9yvTxnvItq?usp=sharing)
